In [2]:

import re, openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,
           'risorse_russia':5,'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,'stabilita_russia':5,
           'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,'stabilita_rotte_cina':5,'stabilita_cina':5,
           'risorse_europa':5,'influenza_diplomatica_europa':5,'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}

GLOBALS = ['nucleare','sanzioni','opinione','defcon']
IRAN_T   = ['risorse_iran','forze_militari_iran','tecnologia_nucleare_iran','stabilita_iran']
COAL_T   = ['risorse_coalizione','influenza_diplomatica_coalizione','tecnologia_avanzata_coalizione','supporto_pubblico_coalizione','stabilita_coalizione']
RUSS_T   = ['risorse_russia','influenza_militare_russia','veto_onu_russia','stabilita_economica_russia','stabilita_russia']
CINA_T   = ['risorse_cina','influenza_commerciale_cina','cyber_warfare_cina','stabilita_rotte_cina','stabilita_cina']
EUR_T    = ['risorse_europa','influenza_diplomatica_europa','aiuti_umanitari_europa','coesione_ue_europa','stabilita_europa']
ALL_FAZ  = IRAN_T+COAL_T+RUSS_T+CINA_T+EUR_T
ALL_T    = GLOBALS+['risorse','stabilita']+ALL_FAZ

FAZIONI  = ['Iran','Coalizione','Russia','Cina','Europa']
RISORSE_KEY = {'Iran':'risorse_iran','Coalizione':'risorse_coalizione','Russia':'risorse_russia','Cina':'risorse_cina','Europa':'risorse_europa'}
STAB_KEY    = {'Iran':'stabilita_iran','Coalizione':'stabilita_coalizione','Russia':'stabilita_russia','Cina':'stabilita_cina','Europa':'stabilita_europa'}
FAZ_MAP  = {'Iran':'iran','Coalizione':'coalizione','Russia':'russia','Cina':'cina','Europa':'europa','Neutrale':None}

TRACK_LABELS = {
    'nucleare':'☢️ Nucleare','sanzioni':'🔒 Sanzioni','opinione':'📣 Opinione','defcon':'🚨 DEFCON',
    'risorse_iran':'💵🇮🇷 Ris.Iran','forze_militari_iran':'⚔️🇮🇷 Forze','tecnologia_nucleare_iran':'🔬🇮🇷 Tecn.','stabilita_iran':'⚖️🇮🇷 Stab.',
    'risorse_coalizione':'💵🇺🇸 Ris.','influenza_diplomatica_coalizione':'🤝🇺🇸 InfDip.','tecnologia_avanzata_coalizione':'💻🇺🇸 Tecn.','supporto_pubblico_coalizione':'📢🇺🇸 Supp.','stabilita_coalizione':'⚖️🇺🇸 Stab.',
    'risorse_russia':'💵🇷🇺 Ris.','influenza_militare_russia':'🎖️🇷🇺 InfMil.','veto_onu_russia':'🏛️🇷🇺 Veto','stabilita_economica_russia':'📊🇷🇺 StabEc.','stabilita_russia':'⚖️🇷🇺 Stab.',
    'risorse_cina':'💵🇨🇳 Ris.','influenza_commerciale_cina':'🏪🇨🇳 InfCom.','cyber_warfare_cina':'🖥️🇨🇳 Cyber','stabilita_rotte_cina':'🚢🇨🇳 Rotte','stabilita_cina':'⚖️🇨🇳 Stab.',
    'risorse_europa':'💵🇪🇺 Ris.','influenza_diplomatica_europa':'🕊️🇪🇺 InfDip.','aiuti_umanitari_europa':'❤️🇪🇺 Aiuti','coesione_ue_europa':'🌐🇪🇺 Coesione','stabilita_europa':'⚖️🇪🇺 Stab.',
}
TRACK_ICONS = {'nucleare':'☢️','sanzioni':'🔒','opinione':'📣','defcon':'🚨','risorse_iran':'💵','forze_militari_iran':'⚔️','tecnologia_nucleare_iran':'🔬','stabilita_iran':'⚖️','risorse_coalizione':'💵','influenza_diplomatica_coalizione':'🤝','tecnologia_avanzata_coalizione':'💻','supporto_pubblico_coalizione':'📢','stabilita_coalizione':'⚖️','risorse_russia':'💵','influenza_militare_russia':'🎖️','veto_onu_russia':'🏛️','stabilita_economica_russia':'📊','stabilita_russia':'⚖️','risorse_cina':'💵','influenza_commerciale_cina':'🏪','cyber_warfare_cina':'🖥️','stabilita_rotte_cina':'🚢','stabilita_cina':'⚖️','risorse_europa':'💵','influenza_diplomatica_europa':'🕊️','aiuti_umanitari_europa':'❤️','coesione_ue_europa':'🌐','stabilita_europa':'⚖️'}
FAZ_FLAG = {'Iran':'🇮🇷','Coalizione':'🇺🇸','Russia':'🇷🇺','Cina':'🇨🇳','Europa':'🇪🇺','Neutrale':'🌐'}

card_re = re.compile(r"\{\s*card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)',\s*op_points:(\d+),\s*deck_type:'([^']+)',\s*description:'([^']*)'",re.DOTALL)
matches = list(card_re.finditer(src))
all_cards = []
for idx,m in enumerate(matches):
    cid,cname,faz,ctype,op,deck,desc = m.groups()
    s=m.start(); e=matches[idx+1].start() if idx+1<len(matches) else len(src)
    block=src[s:e]; em=re.search(r'effects\s*:\s*\{([^}]+)\}',block)
    d={t:0 for t in ALL_T}
    if em:
        for t in ALL_T:
            fn=re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,}}]+)',em.group(1))
            if fn:
                try: d[t]=round(float(eval(f"(lambda v:{fn.group(1).strip().rstrip(',')})(DEFAULT['{t}'])",{'DEFAULT':DEFAULT})),2)
                except: pass
    ris=d.get('risorse',0); stab=d.get('stabilita',0); fk=FAZ_MAP.get(faz)
    if fk:
        d[RISORSE_KEY.get(faz,'risorse_iran')]=(d.get(RISORSE_KEY.get(faz,'risorse_iran'),0)or 0)+ris
        d[STAB_KEY.get(faz,'stabilita_iran')]=(d.get(STAB_KEY.get(faz,'stabilita_iran'),0)or 0)+stab
    else:
        for f2 in FAZIONI:
            d[RISORSE_KEY[f2]]=(d.get(RISORSE_KEY[f2],0)or 0)+ris
            d[STAB_KEY[f2]]=(d.get(STAB_KEY[f2],0)or 0)+stab
    unlocks='unlocks_special:true' in block.replace(' ','')
    parts=[]
    for t in GLOBALS+ALL_FAZ:
        v=d.get(t,0)
        if v: parts.append(f"{TRACK_ICONS.get(t,'?')}{'+' if v>0 else ''}{int(v) if v==int(v) else v}")
    row={'card_id':cid,'card_name':cname,'faction':faz,'card_type':ctype,'op_points':int(op),'deck_type':deck,'description':desc,'unlocks_special':unlocks,'eff_str':' '.join(parts) if parts else '—'}
    row.update({t:d.get(t,0) for t in GLOBALS+ALL_FAZ}); all_cards.append(row)

# Excel
wb=openpyxl.Workbook(); ws=wb.active; ws.title="Tutte le Carte"
FFILL={'Iran':PatternFill('solid',fgColor='FFE0E0'),'Coalizione':PatternFill('solid',fgColor='E0E8FF'),'Russia':PatternFill('solid',fgColor='FFF0E0'),'Cina':PatternFill('solid',fgColor='E0FFE8'),'Europa':PatternFill('solid',fgColor='F0E0FF'),'Neutrale':PatternFill('solid',fgColor='F5F5F5')}
GREEN=PatternFill('solid',fgColor='C8E6C9'); RED=PatternFill('solid',fgColor='FFCDD2')
HD=PatternFill('solid',fgColor='1A1A2E'); HI=PatternFill('solid',fgColor='5C1A1A'); HC=PatternFill('solid',fgColor='1A2A5C')
HR=PatternFill('solid',fgColor='5C3A1A'); HCN=PatternFill('solid',fgColor='1A5C2A'); HE=PatternFill('solid',fgColor='3A1A5C')
HF=Font(bold=True,color='FFFFFF',size=8)

H_INFO=['card_id','card_name','faction','card_type','op','deck','descrizione','speciale']
H_COLS=H_INFO+[TRACK_LABELS[t] for t in GLOBALS+ALL_FAZ]+['Effetti']
ws.append(H_COLS)

SEC=list(range(1,9))+list(range(9,13))+list(range(13,17))+list(range(17,22))+list(range(22,27))+list(range(27,32))+list(range(32,37))+[37]
FILLS_BY_COL={**{i:HD for i in range(1,13)},**{i:HI for i in range(13,17)},**{i:HC for i in range(17,22)},**{i:HR for i in range(22,27)},**{i:HCN for i in range(27,32)},**{i:HE for i in range(32,37)},37:HD}
for i in range(1,len(H_COLS)+1):
    c=ws.cell(1,i); c.fill=FILLS_BY_COL.get(i,HD); c.font=HF; c.alignment=Alignment(horizontal='center',wrap_text=True)

DC=set(range(9,37))
for card in all_cards:
    row_v=([card['card_id'],card['card_name'],card['faction'],card['card_type'],card['op_points'],card['deck_type'],card['description'],'Sì' if card['unlocks_special'] else '']+
           [card.get(t,0) for t in GLOBALS+ALL_FAZ]+[card['eff_str']])
    ws.append(row_v); r=ws.max_row; fz=card['faction']
    for i in range(1,len(H_COLS)+1):
        cell=ws.cell(r,i); cell.alignment=Alignment(vertical='center')
        if i in DC:
            v=cell.value
            if v and v>0: cell.fill=GREEN
            elif v and v<0: cell.fill=RED
        elif i!=len(H_COLS): cell.fill=FFILL.get(fz,PatternFill())

widths=[10,28,12,12,4,9,35,6]+[9]*28+[42]
for i,w in enumerate(widths,1): ws.column_dimensions[get_column_letter(i)].width=w
ws.freeze_panes='A2'; ws.auto_filter.ref=ws.dimensions; ws.row_dimensions[1].height=40

# Foglio 2 — riepilogo
ws2=wb.create_sheet("Riepilogo")
from collections import Counter
ws2.append(['Fazione','Carte','Militare','Diplomatico','Economico','Media','Segreto','Politico','Speciale'])
for c in ws2[1]: c.fill=HD; c.font=Font(bold=True,color='FFFFFF',size=9)
cnt=Counter((c['faction'],c['card_type']) for c in all_cards)
spec=Counter(c['faction'] for c in all_cards if c['deck_type'] in('speciale','speciale_locked'))
tot=Counter(c['faction'] for c in all_cards)
for faz in ['Iran','Coalizione','Russia','Cina','Europa','Neutrale']:
    ws2.append([faz,tot[faz],cnt[(faz,'Militare')],cnt[(faz,'Diplomatico')],cnt[(faz,'Economico')],cnt[(faz,'Media')],cnt[(faz,'Segreto')],cnt[(faz,'Politico')],spec[faz]])
    ws2.cell(ws2.max_row,1).fill=FFILL.get(faz,PatternFill())
for i,w in enumerate([14,7,9,11,9,7,8,8,8],1): ws2.column_dimensions[get_column_letter(i)].width=w

wb.save('/workspace/linea_rossa_carte_completo.xlsx')
print(f"✅ {len(all_cards)} carte × {len(H_COLS)} colonne")


✅ 347 carte × 37 colonne
